# CineEmbed — 00 Colab Setup (one-time per session)

Run this notebook ONCE at the start of each Colab session before running any training notebook.

## New setup flow

**Code** is pulled automatically from GitHub (git clone / git pull) — you no longer need to upload the repo to Drive.

**Artifacts** (feature matrices, checkpoints, results) still live on Google Drive under `MyDrive/CineEmbed/artifacts/`. Drive is the only folder you need to manage manually.

## Required Drive structure (artifacts only — set up manually once)

```
MyDrive/CineEmbed/artifacts/
├── feature_matrix.npz
├── movies_eda_final.csv
├── feature_metadata.json
├── pipeline_version.json
└── director_profile_metadata.json
```

Sub-folders `models/` and `eval/` are created automatically by the training notebooks.

In [ ]:
import os, sys, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Auth: try GitHub token from Colab Secrets (for private repos), fall back to public URL
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f"https://{_token}@github.com/barandincoguz/CineEmbed-.git"
    except Exception:
        REPO_URL = "https://github.com/barandincoguz/CineEmbed-.git"
    REPO_ROOT = Path('/content/cineembed')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        # Pull latest if you've pushed updates from local
        get_ipython().system(f'cd {REPO_ROOT} && git pull -q')

    get_ipython().system(f'pip install -e {REPO_ROOT} -q')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import torch

from cineembed import data, backbone, heads, losses, train, eval as cev

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Repo: {REPO_ROOT}\nArtifacts: {ARTIFACTS}\nDevice: {DEVICE}")

In [ ]:
# Verify artifacts (the repo is auto-cloned above — no need to check it here)
required_artifacts = [
    ARTIFACTS / 'feature_matrix.npz',
    ARTIFACTS / 'movies_eda_final.csv',
    ARTIFACTS / 'feature_metadata.json',
    ARTIFACTS / 'pipeline_version.json',
    ARTIFACTS / 'director_profile_metadata.json',
]

all_ok = True
print('[artifacts]')
for p in required_artifacts:
    ok = p.exists()
    all_ok &= ok
    marker = '\u2713' if ok else '\u2717'
    print(f'  {marker} {p}')

# Create output sub-folders on Drive
(ARTIFACTS / 'models').mkdir(exist_ok=True)
(ARTIFACTS / 'eval').mkdir(exist_ok=True)
print(f"\nCreated/verified {ARTIFACTS / 'models'} and {ARTIFACTS / 'eval'}")

if all_ok:
    print(f"\n\u2705 Artifacts OK.")
else:
    print(f"\n\u274c Missing artifacts — upload the listed files to MyDrive/CineEmbed/artifacts/ and re-run.")

✅ Setup complete. Open 02_train_ae.ipynb next.